First, we import the module for order metric prediction.

In [1]:
import order_metric_prediction as omp
import torch
import h5py
import numpy as np

We load the `HDF5` dictionary containing all order metrics for networks created from the \mathbf{ctn} network and split the data up into training, validation, and test datasets. The algorithm inputs $\beta$, $T_\mathrm{max}$, and $\Delta T$ are the features / neural network inputs. The 42 order metrics are the labels / neural network outputs.

In [2]:
load_data_path="../data/analysis_data/ctn/"
load_filename="all_order_metrics.h5"
neural_network_data_path="../data/neural_networks/ctn/"
data_filename="order_metric_dataset.h5"
test_fraction = 0.15
seed=42

omp.create_dataset_metric_prediction(
    load_data_path=load_data_path,
    load_filename=load_filename,
    neural_network_data_path=neural_network_data_path,
    save_filename=data_filename,
    test_fraction=test_fraction,
    apply_pca=False,
    seed=seed,)

['bond_length_std_vec', 'bond_angle_std_vec', 'dihedral_angle_entropy_vec', 'bond_orientation_entropy_vec', 'coordination_nr_mean_vec', 'coordination_nr_std_vec', 'q_l_mat_values_0', 'q_l_mat_values_1', 'q_l_mat_values_2', 'q_l_mat_values_3', 'q_l_mat_values_4', 'q_l_mat_values_5', 'q_l_mat_values_6', 'q_l_mat_values_7', 'q_l_mat_values_8', 'q_l_mat_values_9', 'q_l_mat_values_10', 'q_l_mat_values_11', 'q_l_mat_values_12', 'q_l_mat_uncertainties_0', 'q_l_mat_uncertainties_1', 'q_l_mat_uncertainties_2', 'q_l_mat_uncertainties_3', 'q_l_mat_uncertainties_4', 'q_l_mat_uncertainties_5', 'q_l_mat_uncertainties_6', 'q_l_mat_uncertainties_7', 'q_l_mat_uncertainties_8', 'q_l_mat_uncertainties_9', 'q_l_mat_uncertainties_10', 'q_l_mat_uncertainties_11', 'q_l_mat_uncertainties_12', 'vertex_homogeneity_metric_vec', 'uncoordinated_neighbor_distance_vec', 'ring_size_mean_vec', 'ring_size_std_vec', 'ring_radius_mean_vec', 'ring_radius_std_vec', 'critical_pore_radius_vec', 'anisotropy_metric_from_struct

We create a second dataset, leveraging correlations in the 42-dimensional order metric data to reduce dimensionality and suppress noise. Here, for each order metric, the data is standardized by removing the mean and scaling to unit variance. Then, a principal component analysis (PCA) is performed, reducing the dimension to the given number of principal components (PCs). This function saves the information about the PCA, like the PC loadings, in the file `order_metric_dataset_pca_information.h5`.

In [3]:
omp.create_dataset_metric_prediction(
    load_data_path=load_data_path,
    load_filename=load_filename,
    neural_network_data_path=neural_network_data_path,
    save_filename=data_filename,
    test_fraction=test_fraction,
    apply_pca=True,
    nr_pca_components=10,
    seed=seed,)

['bond_length_std_vec', 'bond_angle_std_vec', 'dihedral_angle_entropy_vec', 'bond_orientation_entropy_vec', 'coordination_nr_mean_vec', 'coordination_nr_std_vec', 'q_l_mat_values_0', 'q_l_mat_values_1', 'q_l_mat_values_2', 'q_l_mat_values_3', 'q_l_mat_values_4', 'q_l_mat_values_5', 'q_l_mat_values_6', 'q_l_mat_values_7', 'q_l_mat_values_8', 'q_l_mat_values_9', 'q_l_mat_values_10', 'q_l_mat_values_11', 'q_l_mat_values_12', 'q_l_mat_uncertainties_0', 'q_l_mat_uncertainties_1', 'q_l_mat_uncertainties_2', 'q_l_mat_uncertainties_3', 'q_l_mat_uncertainties_4', 'q_l_mat_uncertainties_5', 'q_l_mat_uncertainties_6', 'q_l_mat_uncertainties_7', 'q_l_mat_uncertainties_8', 'q_l_mat_uncertainties_9', 'q_l_mat_uncertainties_10', 'q_l_mat_uncertainties_11', 'q_l_mat_uncertainties_12', 'vertex_homogeneity_metric_vec', 'uncoordinated_neighbor_distance_vec', 'ring_size_mean_vec', 'ring_size_std_vec', 'ring_radius_mean_vec', 'ring_radius_std_vec', 'critical_pore_radius_vec', 'anisotropy_metric_from_struct

Now, we perform hyperparameter tuning on the two datasets. We optimize the network architecture with the number of neurons per hidden layer and the number of layers. Furthermore, the training process is tuned via the batch size and the learning rate. First, we consider the dataset without preprocessing:

In [4]:
checkpoint_path = "../data/neural_networks/ctn/checkpoints/"

omp.hyperparam_tuning(
    weights=omp.get_weights_balanced(),
    nr_layers_tune=[i for i in range(2, 6)],
    nr_neurons_tune=[i for i in range(40, 100)],
    learning_rate_tune=[0.05, 0.1, 0.2 ],
    batch_size_tune=[1, 2, 4],
    weight_decay_tune=[0],
    seed=seed,
    checkpoint_path=checkpoint_path,
    neural_network_data_path =neural_network_data_path,
    data_filename=data_filename,
    nr_inputs=3,
    nr_outputs=42,
    variable_nr_neurons=False,
    max_nr_epochs=10,
    nr_samples_hyperparams=10,)

2026-01-09 11:18:27,202	INFO worker.py:1879 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
2026-01-09 11:18:28,741	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `tune.run(...)`.
2026-01-09 11:18:28,746	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2026-01-09 11:18:28,771	INFO tensorboardx.py:193 -- pip install "ray[tune]" to see TensorBoard files.
2026-01-09 11:18:28,773	WARNING callback.py:136 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not installed. Please make sure you have the latest version of TensorboardX installed: `pip install -U tensorboardx`


(func pid=5505) Epoch:  0
(func pid=5506) [1,   200] loss: 0.020


Trial name,loss,should_checkpoint
train_neural_network_8a9e9_00000,0.00381922,True
train_neural_network_8a9e9_00001,0.00475452,True
train_neural_network_8a9e9_00002,0.00506697,True
train_neural_network_8a9e9_00003,0.00435656,True
train_neural_network_8a9e9_00004,0.00444129,True
train_neural_network_8a9e9_00005,0.00487163,True
train_neural_network_8a9e9_00006,0.00908643,True
train_neural_network_8a9e9_00007,0.0064765,True
train_neural_network_8a9e9_00008,0.00927814,True
train_neural_network_8a9e9_00009,0.00411486,True


(func pid=5515) /home/hemmannf/y/envs/pytorch_env/lib/python3.12/site-packages/ray/train/_internal/session.py:772: RayDeprecationWarning: `ray.train.report` should be switched to `ray.tune.report` when running in a function passed to Ray Tune. This will be an error in the future. See this issue for more context: https://github.com/ray-project/ray/issues/49454
(func pid=5515)   _log_deprecation_warning(
(func pid=5515) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/hemmannf/ray_results/train_neural_network_2026-01-09_11-18-28/train_neural_network_8a9e9_00004_4_batch_size=4,learning_rate=0.1000,nr_layers=2,nr_neurons=55,weight_decay=0_2026-01-09_11-18-29/checkpoint_000000)


(func pid=5515) Validation loss:  0.009052046
(func pid=5513) Epoch:  2 [repeated 13x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(func pid=5506) [2,  1000] loss: 0.001 [repeated 70x across cluster]
(func pid=5506) Validation loss:  0.0055994834 [repeated 9x across cluster]


...

(func pid=5504) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/hemmannf/ray_results/train_neural_network_2026-01-09_11-18-28/train_neural_network_8a9e9_00007_7_batch_size=1,learning_rate=0.2000,nr_layers=5,nr_neurons=96,weight_decay=0_2026-01-09_11-18-29/checkpoint_000007) [repeated 9x across cluster]


(func pid=5503) Validation loss:  0.003985991 [repeated 8x across cluster]


2026-01-09 11:19:33,380	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/hemmannf/ray_results/train_neural_network_2026-01-09_11-18-28' in 0.0092s.
2026-01-09 11:19:33,390	INFO tune.py:1041 -- Total run time: 64.64 seconds (64.58 seconds for the tuning loop).


Best trial config: {'nr_layers': 2, 'nr_neurons': 40, 'learning_rate': 0.2, 'batch_size': 2, 'weight_decay': 0}
Best trial final validation loss: 0.003819220932200551
File found.
Best trial test set loss: 0.004068050534436118


/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:768: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_corr, _ = pearsonr(all_preds[:, i], all_true[:, i])
/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:770: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(all_preds[:, i], all_true[:, i])
/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:768: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pears

Then, we optimize the hyperparameters on the preprocessed data. Here, we use an unweighted mean-squared error loss function to train on the preprocessed order metric data.

In [5]:
omp.hyperparam_tuning(
    weights=None,
    nr_layers_tune=[i for i in range(2, 6)],
    nr_neurons_tune=[i for i in range(40, 100)],
    learning_rate_tune=[0.002, 0.005, 0.01],
    batch_size_tune=[1, 2, 4],
    weight_decay_tune=[0],
    seed=seed,
    checkpoint_path=checkpoint_path,
    neural_network_data_path =neural_network_data_path,
    data_filename="order_metric_dataset_pca_10.h5",
    nr_inputs=3,
    nr_outputs=10,
    max_nr_epochs=10,
    nr_samples_hyperparams=10,)

2026-01-09 11:19:33,681	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2026-01-09 11:19:33,689	WARNING callback.py:136 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not installed. Please make sure you have the latest version of TensorboardX installed: `pip install -U tensorboardx`


(func pid=6090) Epoch:  0
(func pid=6088) [1,   200] loss: 0.352


Trial name,loss,should_checkpoint
train_neural_network_b1505_00000,0.134574,True
train_neural_network_b1505_00001,0.224037,True
train_neural_network_b1505_00002,0.102025,True
train_neural_network_b1505_00003,0.223509,True
train_neural_network_b1505_00004,0.249589,True
train_neural_network_b1505_00005,0.108297,True
train_neural_network_b1505_00006,0.152787,True
train_neural_network_b1505_00007,0.143487,True
train_neural_network_b1505_00008,0.112377,True
train_neural_network_b1505_00009,0.118142,True


(func pid=6090) Validation loss:  0.24817784


(func pid=6090) /home/hemmannf/y/envs/pytorch_env/lib/python3.12/site-packages/ray/train/_internal/session.py:772: RayDeprecationWarning: `ray.train.report` should be switched to `ray.tune.report` when running in a function passed to Ray Tune. This will be an error in the future. See this issue for more context: https://github.com/ray-project/ray/issues/49454
(func pid=6090)   _log_deprecation_warning(
(func pid=6090) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/hemmannf/ray_results/train_neural_network_2026-01-09_11-19-33/train_neural_network_b1505_00000_0_batch_size=4,learning_rate=0.0020,nr_layers=3,nr_neurons=96,weight_decay=0_2026-01-09_11-19-33/checkpoint_000000)
(raylet) I0000 00:00:1767953990.406918    5096 chttp2_transport.cc:1182] ipv4:172.29.28.206:38141: Got goaway [2] err=UNAVAILABLE:GOAWAY received; Error code: 2; Debug Text: Cancelling all calls {created_time:"2026-01-09T11:19:50.406907116+01:00", http2_error:2, grpc_status:14}


...

(func pid=6531) Validation loss:  0.12524188 [repeated 13x across cluster]


(func pid=6531) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/hemmannf/ray_results/train_neural_network_2026-01-09_11-19-33/train_neural_network_b1505_00009_9_batch_size=2,learning_rate=0.0050,nr_layers=4,nr_neurons=93,weight_decay=0_2026-01-09_11-19-33/checkpoint_000004) [repeated 12x across cluster]


(func pid=6531) Epoch:  7 [repeated 5x across cluster]


2026-01-09 11:20:13,229	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/hemmannf/ray_results/train_neural_network_2026-01-09_11-19-33' in 0.0163s.
2026-01-09 11:20:13,241	INFO tune.py:1041 -- Total run time: 39.56 seconds (39.52 seconds for the tuning loop).


Best trial config: {'nr_layers': 4, 'nr_neurons': 87, 'learning_rate': 0.01, 'batch_size': 2, 'weight_decay': 0}
Best trial final validation loss: 0.10202492773532867
File found.
Best trial test set loss: 0.10982915577444011


Now, we train two networks with the optimized hyperparameters and compare their performance using the same weighted mean squared loss function. The optimized hyperparameters may vary depending on the random dataset split into training, validation, and test sets, as well as the randomly chosen hyperparameter combinations. Here, we use the parameters that we found optimal, and we begin with the data without preprocessing.

In [6]:
network_type = "ctn"
nr_layers = 4
nr_neurons = 87
learning_rate = 0.1
nr_epochs = 10

data_filename_test = "order_metric_dataset.h5"
neural_network_filename = f"{network_type}_nr_layers_{nr_layers}_nr_neurons_{nr_neurons}_full.pt"

config = {
    "batch_size": 1,
    "learning_rate": learning_rate,
    "nr_layers": nr_layers,
    "nr_neurons": nr_neurons,
    "weight_decay": 0,}

neural_network = omp.train_neural_network(config, 
              weights=omp.get_weights_balanced(),
              seed=seed, 
              checkpoint_path=checkpoint_path,
              neural_network_data_path=neural_network_data_path,
              data_filename=data_filename,
              nr_inputs=3,
              nr_outputs=42,
              nr_epochs=nr_epochs,
              use_checkpoints=False)

torch.save(neural_network.state_dict(), neural_network_data_path + neural_network_filename)

# load the neural network
neural_network = omp.load_neural_network(nr_inputs=3,
                   nr_outputs=42,
                   nr_neurons=nr_neurons,
                   nr_layers=nr_layers,
                   neural_network_data_path=neural_network_data_path,
                   filename=neural_network_filename)

# Get the test loss
test_loss, pearson_correlations, spearman_correlations, r2_scores, all_preds, all_true = omp.get_test_loss_and_corr(
        neural_network, 
        weights=omp.get_weights_balanced(),
        device="cpu", 
        loss_func=None,
        neural_network_data_path=neural_network_data_path,
        data_filename=data_filename_test,
        nr_outputs=42,
        apply_inverse_pca=False,
        nr_pca_components=None)

print(f"Test loss: {test_loss}")

File found.
Epoch:  0
[1,   200] loss: 0.029
[1,   400] loss: 0.007
[1,   600] loss: 0.004
[1,   800] loss: 0.002
[1,  1000] loss: 0.002
[1,  1200] loss: 0.001
[1,  1400] loss: 0.001
[1,  1600] loss: 0.001
[1,  1800] loss: 0.001
[1,  2000] loss: 0.001
[1,  2200] loss: 0.001
[1,  2400] loss: 0.000
Validation loss:  0.00593119
Epoch:  1
[2,   200] loss: 0.005
[2,   400] loss: 0.003
[2,   600] loss: 0.002
[2,   800] loss: 0.001
[2,  1000] loss: 0.001
[2,  1200] loss: 0.001
[2,  1400] loss: 0.001
[2,  1600] loss: 0.001
[2,  1800] loss: 0.001
[2,  2000] loss: 0.000
[2,  2200] loss: 0.000
[2,  2400] loss: 0.000
Validation loss:  0.004107383
Epoch:  2
[3,   200] loss: 0.005
[3,   400] loss: 0.003
[3,   600] loss: 0.002
[3,   800] loss: 0.001
[3,  1000] loss: 0.001
[3,  1200] loss: 0.001
[3,  1400] loss: 0.001
[3,  1600] loss: 0.001
[3,  1800] loss: 0.000
[3,  2000] loss: 0.000
[3,  2200] loss: 0.000
[3,  2400] loss: 0.000
Validation loss:  0.0046820734
Epoch:  3
[4,   200] loss: 0.004
[4,   4

/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:768: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_corr, _ = pearsonr(all_preds[:, i], all_true[:, i])
/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:770: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(all_preds[:, i], all_true[:, i])
/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:768: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pears

We continue by training the second network on the preprocessed data using only ten principal components.

In [7]:
network_type = "ctn"
nr_layers = 4
nr_neurons = 72
nr_pca_components = 10
learning_rate = 0.005
nr_epochs = 10

data_filename = f"order_metric_dataset_pca_{nr_pca_components}.h5"
neural_network_filename = f"{network_type}_nr_layers_{nr_layers}_nr_neurons_{nr_neurons}_full_pca_{nr_pca_components}.pt"

config = {
    "batch_size": 1,
    "learning_rate": learning_rate,
    "nr_layers": nr_layers,
    "nr_neurons": nr_neurons,
    "weight_decay": 0,}

neural_network = omp.train_neural_network(config, 
              weights=None,
              seed=seed, 
              checkpoint_path=checkpoint_path,
              neural_network_data_path=neural_network_data_path,
              data_filename=data_filename,
              nr_inputs=3,
              nr_outputs=nr_pca_components,
              nr_epochs=nr_epochs,
              use_checkpoints = False)

torch.save(neural_network.state_dict(), neural_network_data_path + neural_network_filename)

# load the neural network
neural_network = omp.load_neural_network(nr_inputs=3,
                   nr_outputs=nr_pca_components,
                   nr_neurons=nr_neurons,
                   nr_layers=nr_layers,
                   neural_network_data_path=neural_network_data_path,
                   filename=neural_network_filename)

# Get the test loss
test_loss, pearson_correlations, spearman_correlations, r2_scores, all_preds, all_true = omp.get_test_loss_and_corr(
        neural_network, 
        weights=omp.get_weights_balanced(),
        device="cpu", 
        loss_func=None,
        neural_network_data_path=neural_network_data_path,
        data_filename=data_filename_test,
        nr_outputs=42,
        apply_inverse_pca=True,
        nr_pca_components=nr_pca_components)

print(f"Test loss: {test_loss}")

File found.
Epoch:  0
[1,   200] loss: 0.343
[1,   400] loss: 0.150
[1,   600] loss: 0.093
[1,   800] loss: 0.057
[1,  1000] loss: 0.046
[1,  1200] loss: 0.038
[1,  1400] loss: 0.029
[1,  1600] loss: 0.025
[1,  1800] loss: 0.019
[1,  2000] loss: 0.015
[1,  2200] loss: 0.014
[1,  2400] loss: 0.013
Validation loss:  0.13878666
Epoch:  1
[2,   200] loss: 0.156
[2,   400] loss: 0.078
[2,   600] loss: 0.052
[2,   800] loss: 0.037
[2,  1000] loss: 0.032
[2,  1200] loss: 0.020
[2,  1400] loss: 0.018
[2,  1600] loss: 0.018
[2,  1800] loss: 0.016
[2,  2000] loss: 0.015
[2,  2200] loss: 0.014
[2,  2400] loss: 0.011
Validation loss:  0.12492952
Epoch:  2
[3,   200] loss: 0.127
[3,   400] loss: 0.074
[3,   600] loss: 0.040
[3,   800] loss: 0.038
[3,  1000] loss: 0.023
[3,  1200] loss: 0.025
[3,  1400] loss: 0.021
[3,  1600] loss: 0.017
[3,  1800] loss: 0.014
[3,  2000] loss: 0.013
[3,  2200] loss: 0.011
[3,  2400] loss: 0.010
Validation loss:  0.11788394
Epoch:  3
[4,   200] loss: 0.120
[4,   400]

/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:768: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_corr, _ = pearsonr(all_preds[:, i], all_true[:, i])
/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:770: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_corr, _ = spearmanr(all_preds[:, i], all_true[:, i])
/mnt/c/Users/HemmannF/OneDrive - Université de Fribourg/structure_analysis/code_photonic_structures/disordered-network-gen-analysis/python_order_metric_prediction/order_metric_prediction.py:768: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pears

We find that the network trained on preprocessed data exhibits a lower test loss than the one trained on the complete set of order metrics. Therefore, in the following, we use the network trained on ten PCs as our best-performing neural network.

The prediction accuracy for the individual order metrics can be assessed using the coefficient of determination $R^2$ returned by the `get_test_loss_and_corr` function, along with a list of other accuracy metrics. Let us save the $R^2$ values for all metrics in an `HDF5` dictionary.

In [8]:
order_metrics_vec = [
        "bond_length_std_vec",
        "bond_angle_std_vec",
        "dihedral_angle_entropy_vec",
        "bond_orientation_entropy_vec",
        "coordination_nr_mean_vec",
        "coordination_nr_std_vec",
        "q_l_mat_values",
        "q_l_mat_uncertainties",
        "vertex_homogeneity_metric_vec",
        "uncoordinated_neighbor_distance_vec",
        "ring_size_mean_vec",
        "ring_size_std_vec",
        "ring_radius_mean_vec",
        "ring_radius_std_vec",
        "critical_pore_radius_vec",
        "anisotropy_metric_from_structure_factor_vec",
        "anisotropy_metric_from_structure_factor_bonds_vec",
        "hyperuniformity_alpha_vec"]

neural_network_data_path = "../data/neural_networks/ctn/"

r2_output_filename = f"{network_type}_nr_layers_{nr_layers}_nr_neurons_{nr_neurons}_r2_scores_pca_{nr_pca_components}.h5"
with h5py.File(neural_network_data_path + r2_output_filename, 'w') as hf:
    for i in range(6):
        hf.create_dataset(order_metrics_vec[i], data=r2_scores[i])
    
    for i in range(8, 18):
        hf.create_dataset(order_metrics_vec[i], data=r2_scores[i+24])

    hf.create_dataset("q_l_mat_values", data=np.array(r2_scores[6:19]))
    hf.create_dataset("q_l_mat_uncertainties", data=np.array(r2_scores[19:32]))

We can now use the trained neural network to predict order metrics. To scale temperatures in units of the melting temperature $\beta$ -dependent $T_\mathrm{melt}$, we load the dictionary `ctn_t_melt_vec.h5` which was created by the Julia code.

In [9]:
t_melt_filename = f"{network_type}_t_melt_vec.h5"

with h5py.File(load_data_path+t_melt_filename, 'r') as hf:
    bond_bending_const_vec = hf["bond_bending_const_vec"][:]
    t_melt_vec = hf["t_melt_vec"][:]
    t_gradient_vec = hf["t_gradient_vec"][:]
    t_max_over_t_melt_vec = hf["t_max_over_t_melt_vec"][:]

# print bond_bending_const_vec and t_melt_vec
print("Bond bending constant vector:")
print(bond_bending_const_vec)
print("T_melt vector:")
print(t_melt_vec)
print("T_gradient vector:")
print(t_gradient_vec)

# create an array with the maximal temperature values
t_max_arr = t_max_over_t_melt_vec[:, np.newaxis] * t_melt_vec[np.newaxis, :]

# create a 3d array with dimensions (len(bond_bending_const_vec), len(t_max_arr), len(t_gradient_vec))
# and fill it with zeros
predictions_array = np.zeros((len(t_max_over_t_melt_vec), len(bond_bending_const_vec), 42))

# Get model's device and dtype
device = next(neural_network.parameters()).device
dtype = next(neural_network.parameters()).dtype  # likely torch.float32

# Calculate the number of data points
num_data_points = len(bond_bending_const_vec) * len(t_max_over_t_melt_vec)
print(f"Total number of data points to predict: {num_data_points}")

current_data_point = 0

# loop through all combinations of bond_bending_const_vec, t_max_arr, 
# t_gradient_vec and predict the order metrics
for i, t_max_over_t_melt in enumerate(t_max_over_t_melt_vec):
    for j, bond_bending_const in enumerate(bond_bending_const_vec):
            
        t_gradient = t_gradient_vec[j]
        t_max = t_max_over_t_melt * t_melt_vec[j]
        test_input = torch.tensor(
            [[float(bond_bending_const), float(t_max), float(t_gradient)]],
            dtype=dtype,
            device=device
        )
        with torch.no_grad():
            outputs = neural_network(test_input)
        scaler, pca = omp.load_scaler_and_pca(
            filename=data_filename,
            neural_network_data_path=neural_network_data_path)
        prediction = pca.inverse_transform(outputs.detach().numpy())
        prediction = scaler.inverse_transform(prediction)
        # for the first one, print prediction
        if i == 0 and j == 0:
            print(f"Prediction for bond_bending_const={bond_bending_const}, t_max={t_max}, t_gradient={t_gradient}: {prediction}")
        predictions_array[i, j, :] = prediction
        # print progress every 200 data points
        current_data_point += 1
        if current_data_point % 200 == 0:
            print(f"Predicted {current_data_point} / {num_data_points} data points.")

# save the predictions_array to an h5 file
with h5py.File(neural_network_data_path+f"{network_type}_predictions_nr_layers_{nr_layers}_nr_neurons_{nr_neurons}_full_pca_{nr_pca_components}.h5", 'w') as hf:
    hf.create_dataset("predictions_array", data=predictions_array)
    hf.create_dataset("bond_bending_const_vec", data=bond_bending_const_vec)
    hf.create_dataset("t_max_arr", data=t_max_arr)
    hf.create_dataset("t_gradient_vec", data=t_gradient_vec)
    hf.create_dataset("t_melt_vec", data=t_melt_vec)
    hf.create_dataset("t_max_over_t_melt_vec", data=t_max_over_t_melt_vec)

Bond bending constant vector:
[ 0.   0.2  0.4  0.6  0.8  1.   1.2  1.4  1.6  1.8  2.   2.2  2.4  2.6
  2.8  3.   3.2  3.4  3.6  3.8  4.   4.2  4.4  4.6  4.8  5.   5.2  5.4
  5.6  5.8  6.   6.2  6.4  6.6  6.8  7.   7.2  7.4  7.6  7.8  8.   8.2
  8.4  8.6  8.8  9.   9.2  9.4  9.6  9.8 10. ]
T_melt vector:
[4.14991703e-04 1.00285769e-01 2.33087467e-01 3.57071208e-01
 4.73915063e-01 6.28724058e-01 8.33854272e-01 9.95205636e-01
 1.17738669e+00 1.40609073e+00 1.66809581e+00 1.87372599e+00
 2.02651320e+00 2.26691733e+00 2.57712307e+00 2.78103989e+00
 2.97849316e+00 3.06541862e+00 3.42231682e+00 3.97351148e+00
 4.18714952e+00 4.65169672e+00 4.88256919e+00 4.91158468e+00
 4.98713427e+00 5.57418244e+00 5.98938663e+00 5.81249696e+00
 5.97459827e+00 6.60910540e+00 7.05176316e+00 7.52888209e+00
 7.55159960e+00 7.54140676e+00 7.49304906e+00 6.97452642e+00
 8.75381886e+00 8.64873621e+00 8.83377416e+00 9.35048074e+00
 8.10745635e+00 9.34623220e+00 9.82830973e+00 1.01874270e+01
 1.04495558e+01 1.003290